In [0]:
-- ============================================================================
-- PROYECTO: Comercializadora Andina - Ingesta y Procesamiento con Spark SQL
-- ARQUITECTURA: Multi-Environment Medallion (Entorno DEV)
-- CATALOGO: dev_practica_ui
-- FECHA: 2026-11-03
-- ============================================================================


-- ==========================================
-- SECCIÓN 2: DDL DE ARQUITECTURA
-- ==========================================
-- 1. Crear el Catálogo principal de Desarrollo
CREATE CATALOG IF NOT EXISTS dev_practica_ui;
USE CATALOG dev_practica_ui;

-- 2. Crear las capas Medallion
CREATE SCHEMA IF NOT EXISTS raw    COMMENT 'Almacenamiento de archivos crudos en Volúmenes';
CREATE SCHEMA IF NOT EXISTS bronze COMMENT 'Tablas Delta ingestadas 1:1 desde la capa Raw';
CREATE SCHEMA IF NOT EXISTS silver COMMENT 'Tablas Delta limpias, tipadas y desduplicadas';
CREATE SCHEMA IF NOT EXISTS gold   COMMENT 'Vistas y agregaciones finales para KPIs de Negocio';

-- 3. Crear el Volumen gestionado en Raw
CREATE VOLUME IF NOT EXISTS dev_practica_ui.raw.archivos_ui
COMMENT 'Contenedor para archivos CSV, JSON y Parquet fuente';


-- ==========================================
-- SECCIÓN 6: INGESTA RAW TO BRONZE
-- ==========================================
USE CATALOG dev_practica_ui;
USE SCHEMA bronze;

-- Ingesta de Productos (CSV)
CREATE OR REPLACE TABLE dev_practica_ui.bronze.productos AS
SELECT 
    *
FROM read_files(
    '/Volumes/dev_practica_ui/raw/archivos_ui/productos.csv'
    ,format => 'csv'
    ,header => true
    ,inferSchema => true
);

-- Ingesta de Ventas (JSON)
CREATE OR REPLACE TABLE dev_practica_ui.bronze.ventas AS
SELECT 
    *
FROM read_files(
    '/Volumes/dev_practica_ui/raw/archivos_ui/ventas_2025.json'
    ,format => 'json'
);

-- Ingesta de Vendedores (Parquet)
CREATE OR REPLACE TABLE dev_practica_ui.bronze.vendedores AS
SELECT 
    *
FROM read_files(
    '/Volumes/dev_practica_ui/raw/archivos_ui/vendedores.parquet'
    ,format => 'parquet'
);


-- ==========================================
-- SECCIÓN 7: METADATOS Y RESTRICCIONES (DDL)
-- ==========================================
-- Comentarios a nivel de tabla
COMMENT ON TABLE dev_practica_ui.bronze.productos IS 'Catálogo de productos disponibles';
COMMENT ON TABLE dev_practica_ui.bronze.vendedores IS 'Fuerza de ventas por ciudad';
COMMENT ON TABLE dev_practica_ui.bronze.ventas IS 'Transacciones de ventas del año 2025';

-- Comentarios de columnas
ALTER TABLE dev_practica_ui.bronze.productos ALTER COLUMN id_producto COMMENT 'Identificador único del producto (PK)';
ALTER TABLE dev_practica_ui.bronze.productos ALTER COLUMN precio_lista COMMENT 'Precio unitario en pesos colombianos (COP)';

-- Restricciones CHECK y NOT NULL
ALTER TABLE dev_practica_ui.bronze.productos ADD CONSTRAINT chk_precio_positivo CHECK (precio_lista > 0);
ALTER TABLE dev_practica_ui.bronze.productos ALTER COLUMN id_producto SET NOT NULL;


-- ==========================================
-- SECCIÓN 8: CONSULTAS Y AGRUPACIONES
-- ==========================================
-- Agregación por categorías
SELECT categoria                             -- columna de agrupación
    ,count(*)                 AS productos -- cuántos productos hay en el grupo
    ,round(avg(precio_lista)) AS precio_promedio
    ,max(precio_lista)        AS precio_maximo
FROM dev_practica_ui.bronze.productos
GROUP BY categoria                             -- una fila por categoría
ORDER BY productos DESC;


WITH cte_ventas_consolidadas AS (
    SELECT 
        ven.ciudad
        ,p.categoria
        ,v.cantidad
        ,(v.precio_unitario * v.cantidad) as monto_total
    FROM dev_practica_ui.bronze.ventas AS v
    INNER JOIN dev_practica_ui.bronze.productos AS p ON v.id_producto = p.id_producto
    INNER JOIN dev_practica_ui.bronze.vendedores AS ven ON v.id_vendedor = ven.id_vendedor
)
SELECT ciudad
    ,categoria
    ,COUNT(1)           AS total_transacciones
    ,SUM(cantidad)      AS total_unidades
    ,ROUND(SUM(monto_total))  AS venta_total
FROM cte_ventas_consolidadas
GROUP BY ciudad,categoria
ORDER BY venta_total DESC;




USE CATALOG dev_practica_ui;
USE SCHEMA bronze;

-- Unir las tres tablas para analizar transacciones con detalle
SELECT v.id_venta
    ,v.fecha_venta
    ,p.nombre_producto
    ,p.categoria
    ,p.precio_lista
    ,v.cantidad
    ,v.precio_unitario
    ,ven.nombre_vendedor
    ,ven.ciudad
FROM dev_practica_ui.bronze.ventas AS v
INNER JOIN dev_practica_ui.bronze.productos AS p ON v.id_producto = p.id_producto
INNER JOIN dev_practica_ui.bronze.vendedores AS ven ON v.id_vendedor = ven.id_vendedor
LIMIT 100;